# XGBoost: Missing Value Handling with and without Imputation

This notebook compares how XGBoost performs with and without missing value imputation using the Titanic dataset.

**Objective:** Evaluate whether manual imputation improves model performance over XGBoost's internal handling of missing values.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [2]:
# Load your Titanic dataset (make sure train.csv is uploaded)
df = pd.read_csv('train.csv')
features = ['Age', 'Fare', 'Sex', 'Embarked']
X = df[features]
y = df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Option A: Without Imputation (Let XGBoost Handle NaNs)

In [3]:
cat_features = ['Sex', 'Embarked']
num_features = ['Age', 'Fare']

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_no_impute = ColumnTransformer([
    ('cat', cat_pipeline, cat_features)
], remainder='passthrough')  # Leave numerical features as-is (can have NaNs)

model_no_impute = Pipeline([
    ('preprocessor', preprocessor_no_impute),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

model_no_impute.fit(X_train, y_train)
y_pred_no_impute = model_no_impute.predict(X_test)
acc_no_impute = accuracy_score(y_test, y_pred_no_impute)
print(f"✅ Accuracy WITHOUT imputation: {acc_no_impute:.4f}")

✅ Accuracy WITHOUT imputation: 0.7654


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:44:00] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


## Option B: With Median Imputation

In [4]:
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))
])

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

preprocessor_with_impute = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

model_with_impute = Pipeline([
    ('preprocessor', preprocessor_with_impute),
    ('xgb', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

model_with_impute.fit(X_train, y_train)
y_pred_with_impute = model_with_impute.predict(X_test)
acc_with_impute = accuracy_score(y_test, y_pred_with_impute)
print(f"✅ Accuracy WITH median imputation: {acc_with_impute:.4f}")

✅ Accuracy WITH median imputation: 0.7877


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [12:44:00] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [5]:
print("\n🔍 Accuracy Comparison:")
print(f"XGBoost without imputation: {acc_no_impute:.4f}")
print(f"XGBoost with median imputation: {acc_with_impute:.4f}")


🔍 Accuracy Comparison:
XGBoost without imputation: 0.7654
XGBoost with median imputation: 0.7877
